# SetFit vs. fine-tuning tradicional

Este notebook compara, nos mesmos cinco folds estratificados:

- `joeddav/xlm-roberta-large-xnli` com SetFit e com o fine-tuning tradicional já executado;
- `neuralmind/bert-base-portuguese-cased` com SetFit e com o fine-tuning tradicional já executado.

SetFit não reutiliza a cabeça de classificação XNLI. O encoder do checkpoint gera embeddings, esses embeddings são ajustados por aprendizagem contrastiva e uma regressão logística aprende as três classes. Cada fold começa novamente no checkpoint original.

Os resultados tradicionais são carregados de `data/model_comparison/` e só são aceites quando os UIDs, labels e folds coincidem com os dados atuais. As únicas saídas visíveis são as duas tabelas finais.

Antes de executar pela primeira vez, atualiza o ambiente com `pip install -r requirements.txt`.

In [ ]:
import json
import logging
import os
import sys
import tempfile
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from packaging.version import Version

try:
    from datasets import Dataset
    from setfit import SetFitModel, Trainer as SetFitTrainer
    from setfit import TrainingArguments as SetFitTrainingArguments
except ImportError as error:
    raise RuntimeError('Instala as dependências com: pip install -r requirements.txt') from error

if Version(version('setfit')) < Version('1.2'):
    raise RuntimeError('Este notebook requer setfit>=1.2 para compatibilidade com Transformers 5.')

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Executa o notebook na raiz do projeto ou dentro de research/.')
sys.path.insert(0, str(ROOT))

from src.email_data import LABELS
from src.finetuning import (
    DEFAULT_EMAILS_DIR,
    DEFAULT_EXCEL,
    clear_device_cache,
    dataset_fingerprint,
    load_examples,
    scores,
    stratified_folds,
)

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
for logger_name in ('setfit', 'sentence_transformers', 'transformers'):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

if not torch.cuda.is_available():
    raise RuntimeError('Este notebook requer uma GPU NVIDIA com CUDA.')

DEVICE = 'cuda'
SEED = 42
N_FOLDS = 5
SETFIT_EPOCHS = 3
SETFIT_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 8
BODY_LEARNING_RATE = 2e-5
MAX_LENGTH = 512

SETFIT_MODELS = {
    'XLM-R large XNLI (SetFit)': 'joeddav/xlm-roberta-large-xnli',
    'BERTimbau base (SetFit)': 'neuralmind/bert-base-portuguese-cased',
}
TRADITIONAL_NAMES = {
    'XLM-R large XNLI (fine-tuning)': 'XLM-R large XNLI (fine-tuning tradicional)',
    'BERTimbau base (fine-tuning)': 'BERTimbau base (fine-tuning tradicional)',
}
COMPARISON_ORDER = [
    'XLM-R large XNLI (fine-tuning tradicional)',
    'XLM-R large XNLI (SetFit)',
    'BERTimbau base (fine-tuning tradicional)',
    'BERTimbau base (SetFit)',
]

TRADITIONAL_DIR = ROOT / 'data/model_comparison'
OUTPUT_DIR = ROOT / 'data/setfit_comparison'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
examples = load_examples(DEFAULT_EXCEL, [DEFAULT_EMAILS_DIR])
folds = stratified_folds(examples, N_FOLDS, SEED)
fold_by_uid = {
    uid: fold_index
    for fold_index, fold in enumerate(folds, start=1)
    for uid, _text, _label in fold
}
truth_by_uid = {uid: label for uid, _text, label in examples}

required_files = [
    TRADITIONAL_DIR / 'summary.csv',
    TRADITIONAL_DIR / 'recall_by_class.csv',
    TRADITIONAL_DIR / 'predictions.csv',
    TRADITIONAL_DIR / 'configuration.json',
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Executa primeiro compare_base_models_and_xnli.ipynb. Faltam: ' + ', '.join(missing_files)
    )

traditional_summary = pd.read_csv(required_files[0], index_col='Modelo')
traditional_recall = pd.read_csv(required_files[1], index_col='Modelo')
traditional_predictions = pd.read_csv(required_files[2], dtype={'uid': str})
traditional_config = json.loads(required_files[3].read_text(encoding='utf-8'))

missing_models = set(TRADITIONAL_NAMES) - set(traditional_summary.index)
if missing_models:
    raise ValueError(f'Faltam resultados tradicionais para: {sorted(missing_models)}')

stored_folds = {str(uid): int(fold) for uid, fold in traditional_config['folds_by_uid'].items()}
if stored_folds != fold_by_uid:
    raise ValueError('Os folds dos resultados tradicionais não coincidem com os dados atuais.')

for source_name in TRADITIONAL_NAMES:
    rows = traditional_predictions[traditional_predictions['model'] == source_name]
    if len(rows) != len(examples) or set(rows['uid']) != set(truth_by_uid):
        raise ValueError(f'Cobertura de UIDs inválida nos resultados de {source_name}.')
    stored_truth = dict(zip(rows['uid'], rows['label_real']))
    stored_fold_by_uid = dict(zip(rows['uid'], rows['fold'].astype(int)))
    if stored_truth != truth_by_uid or stored_fold_by_uid != fold_by_uid:
        raise ValueError(f'Labels ou folds desatualizados nos resultados de {source_name}.')

In [ ]:
def make_dataset(rows):
    return Dataset.from_dict({
        'text': [text for _uid, text, _label in rows],
        'label': [LABELS.index(label) for _uid, _text, label in rows],
    })


def setfit_cross_validation(display_name, model_name):
    predictions = []

    for fold_index, test_examples in enumerate(folds, start=1):
        test_uids = {uid for uid, _text, _label in test_examples}
        train_examples = [example for example in examples if example[0] not in test_uids]
        fold_seed = SEED + fold_index - 1

        train_dataset = make_dataset(train_examples)
        with tempfile.TemporaryDirectory(prefix=f'setfit-fold-{fold_index}-') as temporary:
            model = SetFitModel.from_pretrained(
                model_name,
                labels=list(LABELS),
                device=DEVICE,
            )
            model.model_body.max_seq_length = MAX_LENGTH
            args = SetFitTrainingArguments(
                output_dir=temporary,
                batch_size=SETFIT_BATCH_SIZE,
                num_epochs=SETFIT_EPOCHS,
                body_learning_rate=BODY_LEARNING_RATE,
                max_length=MAX_LENGTH,
                sampling_strategy='oversampling',
                use_amp=True,
                show_progress_bar=False,
                logging_strategy='no',
                save_strategy='no',
                report_to='none',
                seed=fold_seed,
            )
            trainer = SetFitTrainer(
                model=model,
                args=args,
                train_dataset=train_dataset,
            )
            trainer.train()
            raw_predictions = model.predict(
                [text for _uid, text, _label in test_examples],
                batch_size=EVAL_BATCH_SIZE,
                use_labels=True,
                show_progress_bar=False,
            )

            for (uid, _text, real_label), predicted_label in zip(test_examples, raw_predictions):
                predictions.append({
                    'model': display_name,
                    'uid': uid,
                    'fold': fold_index,
                    'label_real': real_label,
                    'label_prevista': str(predicted_label),
                })

            del trainer, model, train_dataset, raw_predictions
            clear_device_cache()

    if len(predictions) != len(examples) or len({row['uid'] for row in predictions}) != len(examples):
        raise RuntimeError(f'A avaliação de {display_name} não cobriu cada UID exatamente uma vez.')
    invalid_labels = {row['label_prevista'] for row in predictions} - set(LABELS)
    if invalid_labels:
        raise RuntimeError(f'{display_name} produziu labels inválidas: {sorted(invalid_labels)}')
    return sorted(predictions, key=lambda row: row['uid'])

In [ ]:
setfit_results = {
    display_name: setfit_cross_validation(display_name, model_name)
    for display_name, model_name in SETFIT_MODELS.items()
}

setfit_summary_rows = []
setfit_recall_rows = []
setfit_prediction_rows = []
for model_name, rows in setfit_results.items():
    truth = [row['label_real'] for row in rows]
    predicted = [row['label_prevista'] for row in rows]
    model_scores = scores(truth, predicted)
    setfit_summary_rows.append({
        'Modelo': model_name,
        'Accuracy': model_scores['accuracy'],
        'Recall macro': model_scores['recall_macro'],
    })
    setfit_recall_rows.append({
        'Modelo': model_name,
        **{
            f'Recall — {label}': model_scores['per_class'][label]['recall']
            for label in LABELS
        },
    })
    setfit_prediction_rows.extend(rows)

traditional_summary_selected = traditional_summary.loc[list(TRADITIONAL_NAMES)].rename(index=TRADITIONAL_NAMES)
traditional_recall_selected = traditional_recall.loc[list(TRADITIONAL_NAMES)].rename(index=TRADITIONAL_NAMES)
setfit_summary = pd.DataFrame(setfit_summary_rows).set_index('Modelo')
setfit_recall = pd.DataFrame(setfit_recall_rows).set_index('Modelo')
summary_table = pd.concat([traditional_summary_selected, setfit_summary]).loc[COMPARISON_ORDER]
recall_table = pd.concat([traditional_recall_selected, setfit_recall]).loc[COMPARISON_ORDER]

traditional_predictions_selected = traditional_predictions[
    traditional_predictions['model'].isin(TRADITIONAL_NAMES)
].copy()
traditional_predictions_selected['model'] = traditional_predictions_selected['model'].map(TRADITIONAL_NAMES)
combined_predictions = pd.concat([
    traditional_predictions_selected,
    pd.DataFrame(setfit_prediction_rows),
], ignore_index=True)

summary_table.to_csv(OUTPUT_DIR / 'summary.csv', encoding='utf-8-sig')
recall_table.to_csv(OUTPUT_DIR / 'recall_by_class.csv', encoding='utf-8-sig')
combined_predictions.to_csv(OUTPUT_DIR / 'predictions.csv', index=False, encoding='utf-8-sig')
_ = (OUTPUT_DIR / 'configuration.json').write_text(
    json.dumps({
        'dataset_fingerprint': dataset_fingerprint(examples),
        'seed': SEED,
        'folds': N_FOLDS,
        'setfit_epochs': SETFIT_EPOCHS,
        'setfit_batch_size': SETFIT_BATCH_SIZE,
        'body_learning_rate': BODY_LEARNING_RATE,
        'max_length': MAX_LENGTH,
        'setfit_models': SETFIT_MODELS,
        'traditional_models': TRADITIONAL_NAMES,
        'folds_by_uid': fold_by_uid,
    }, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

## Resultados globais

In [ ]:
display(summary_table.style.format('{:.3f}'))

## Recall por classe

In [ ]:
display(recall_table.style.format('{:.3f}'))